# Lab 06: Cloud Scheduling with Reinforcement Learning

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>

This notebook covers:
- **CloudSchedulingEnv** for **resource provisioning and task scheduling**
- **Student TODO sections** for RL agents, training loops, and evaluation
- **Baseline testing** for scheduling decisions and performance analysis

</div>

### Imports and Setup

The following cell configures the runtime and defines shared utilities used across all parts of this lab.

In [ ]:
import copy
import os
import random
from collections import deque

import gymnasium as gym
import numpy as np
from gymnasium import spaces

In [ ]:
# Additional imports for RL agents
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

## Part 1: Cloud Scheduling Environment and Problem Setup

### 1.1. Task Model and Environment Goals

Each line in the workload file is converted into one independent task with CPU, RAM, and disk requirements.
The environment keeps track of the ready queue, running tasks, completed tasks, and rejected tasks.
Students should first understand how the environment represents workload items before attempting to modify the scheduling logic.

The main goal of this section is to show how cloud resource provisioning can be expressed as a reinforcement learning problem.

### 1.2. Environment API, State Vector, and Scheduling Actions

`CloudSchedulingEnv` follows the Gymnasium API by exposing `reset()` and `step()` methods.
The observation vector combines summary statistics, the current task description, and the flattened server resource state.
Each action selects a server and VM pair, while the wait action lets the scheduler defer the decision for one time step.
The reward encourages valid placements, discourages waiting, and penalizes invalid scheduling decisions.

In [ ]:
"""
Simple cloud scheduling environment with independent tasks, multiple servers and VMs, 
and a simple reward function based on task completion and deadlines.
"""

class CloudTask(object):
    """Represent a single independent task."""

    def __init__(self, job_id, task_index, cpu_demand, ram_demand, disk_demand, order_index):
        self.job_id = str(job_id)
        self.task_index = int(float(task_index))
        self.cpu_demand = float(cpu_demand)
        self.ram_demand = float(ram_demand)
        self.disk_demand = float(disk_demand)
        self.order_index = int(order_index)

        # Derive a synthetic runtime and deadline from the task size.
        # task runtime is proportional to the sum of resource demands, scaled by 20.0, with a minimum of 1.0.
        # 20.0 is an arbitrary scaling factor to make runtimes more realistic for the given resource demands.
        self.runtime = max(1.0, (self.cpu_demand + self.ram_demand + self.disk_demand) * 20.0)
        # task deadline is set to 4 times the runtime plus an additional 10.0 time units, to allow for some scheduling flexibility.
        self.deadline = self.runtime * 4.0 + 10.0

        self.status = "pending"
        self.ready_time = None
        self.start_time = None
        self.finish_time = None
        self.assigned_location = None

In [ ]:

class CloudSchedulingEnv(gym.Env):
    """Gymnasium-style environment for scheduling independent tasks on servers and VMs.

    Action semantics:
    - ``0`` to ``num_servers * num_vms_per_server - 1`` selects a server and VM pair.
    - ``wait_action`` means do nothing for one time step.
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, scale="small", workload_path="dataset.txt", num_tasks=None,
                 num_servers=300, num_vms_per_server=5):
        super().__init__()
        self.scale = scale
        self.workload_path = workload_path
        self.num_tasks = num_tasks
        self.num_servers = int(num_servers)
        self.num_vms_per_server = int(num_vms_per_server)

        # map flat action indices to (server_index, vm_index) pairs
        self.action_map = self._build_action_map()
        # the action space includes all server-VM pairs plus one additional "wait" action
        self.action_space_size = len(self.action_map) + 1
        # the last action index is reserved for the "wait" action
        self.wait_action = self.action_space_size - 1

        # job templates loaded from the workload file, each representing an independent task
        self.job_templates = self._load_workload(self.workload_path, self.num_tasks)
        self.total_tasks = len(self.job_templates)
        self.max_steps = max(1, self.total_tasks * 4)
        self.max_runtime = max((task.runtime for task in self.job_templates), default=1.0)
        self.max_deadline = max((task.deadline for task in self.job_templates), default=1.0)
        
        # state size = summary features + current task features + resource features for all server-VM pairs
        self.state_size = self._calculate_state_size()
        self.action_space = spaces.Discrete(self.action_space_size)
        self.observation_space = spaces.Box(
            low=0.0,
            high=np.finfo(np.float32).max,
            shape=(self.state_size,),
            dtype=np.float32,
        )

        self.reset()

    def _build_action_map(self):
        """Flatten server and VM indices into a single action list."""
        action_map = []
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                action_map.append((server_index, vm_index))
        return action_map

    def _calculate_state_size(self):
        """Return the length of the observation vector."""
        summary_features = 3
        task_features = 6
        resource_features = len(self.action_map) * 3
        return summary_features + task_features + resource_features

    def _load_workload(self, workload_path, max_tasks=None):
        """Load workload data and convert every line into one independent task."""
        if not workload_path or not os.path.exists(workload_path):
            raise FileNotFoundError("Workload file not found: {0}".format(workload_path))

        tasks = []
        order_index = 0

        with open(workload_path, "r") as file_obj:
            for raw_line in file_obj:
                line = raw_line.strip()
                if not line or line.startswith("#") or line.startswith("//"):
                    continue

                tokens = line.split()
                if len(tokens) < 7:
                    continue

                task = CloudTask(
                    job_id=tokens[1],
                    task_index=tokens[2],
                    cpu_demand=tokens[4],
                    ram_demand=tokens[5],
                    disk_demand=tokens[6],
                    order_index=order_index,
                )
                tasks.append(task)
                order_index += 1

                if max_tasks is not None and len(tasks) >= int(max_tasks):
                    break

        return tasks

    def _build_server_resources(self):
        """Create the resource matrix for all servers and VMs."""
        # For simplicity, we assume each VM has the same capacity, and the total capacity of a server is normalized to 1.0 for each resource type.
        # There are 3 resource types: CPU, RAM, and Disk. Each VM gets an equal share of the server's total capacity.
        vm_capacity = 1.0 / self.num_vms_per_server
        return [
            [[vm_capacity, vm_capacity, vm_capacity] for _ in range(self.num_vms_per_server)]
            for _ in range(self.num_servers)
        ]

    def _build_running_task_slots(self):
        """Create the running-task list for every VM."""
        return [[[] for _ in range(self.num_vms_per_server)] for _ in range(self.num_servers)]

    def _decode_action(self, action):
        """Map a flat action index back to server and VM indices."""
        if action is None:
            return None

        action_index = int(action)
        if action_index == self.wait_action:
            return None
        if action_index < 0 or action_index >= len(self.action_map):
            return None
        return self.action_map[action_index]

    def _can_place_task(self, task, server_index, vm_index):
        """Check whether the selected VM can run the current task."""
        if server_index is None:
            return False
        if server_index < 0 or server_index >= self.num_servers:
            return False
        if vm_index < 0 or vm_index >= self.num_vms_per_server:
            return False

        vm_resources = self.resources[server_index][vm_index]
        # Check if the VM has enough resources to run the task and if it can meet the task's deadline.
        has_capacity = (
            vm_resources[0] >= task.cpu_demand and
            vm_resources[1] >= task.ram_demand and
            vm_resources[2] >= task.disk_demand
        )
        # Check if the task can finish before its deadline if started now.
        will_meet_deadline = self.current_time + task.runtime <= task.deadline
        return has_capacity and will_meet_deadline

    def _allocate_task(self, task, server_index, vm_index):
        """Reserve resources and mark the task as running."""
        vm_resources = self.resources[server_index][vm_index]
        # If the task is allocated to this VM, we need to reduce the available resources accordingly.
        vm_resources[0] -= task.cpu_demand
        vm_resources[1] -= task.ram_demand
        vm_resources[2] -= task.disk_demand

        task.status = "running"
        task.start_time = self.current_time
        task.finish_time = self.current_time + task.runtime
        task.assigned_location = (server_index, vm_index)
        self.running_tasks[server_index][vm_index].append(task)

    def _release_completed_tasks(self):
        """Release tasks whose finish time has passed."""
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                still_running = []
                for task in self.running_tasks[server_index][vm_index]:
                    if task.finish_time is not None and task.finish_time <= self.current_time:
                        # If the task has finished, mark it as completed.
                        task.status = "finished"
                        self.completed_tasks += 1
                        # And, release the resources back to the VM.
                        self.resources[server_index][vm_index][0] += task.cpu_demand
                        self.resources[server_index][vm_index][1] += task.ram_demand
                        self.resources[server_index][vm_index][2] += task.disk_demand
                    else:
                        still_running.append(task)
                self.running_tasks[server_index][vm_index] = still_running

    def _task_wait_time(self, task):
        """Return how long the task has waited in the queue."""
        if task.ready_time is None:
            return 0.0
        return max(0.0, self.current_time - task.ready_time)

    def _calculate_reward(self, task, success, reason=None):
        """Compute reward: encourages fast scheduling, penalises waiting and rejection."""
        if success:
            base_reward = 1.0
            # Urgency bonus: reward more when deadline headroom is large
            time_margin   = task.deadline - (self.current_time + task.runtime)
            urgency_bonus = 0.5 * max(0.0, time_margin / task.deadline) if task.deadline > 0 else 0.0
            # Waiting penalty: small negative proportional to queue wait
            wait_time     = self._task_wait_time(task)
            wait_penalty  = -0.01 * (wait_time / float(max(1, self.max_steps)))
            return base_reward + urgency_bonus + wait_penalty
        else:
            if reason == "wait":
                return -0.05   # light penalty for deferring
            return -1.0        # heavy penalty for rejection

    def _current_task(self):
        """Return the next task to be scheduled."""
        return self.ready_queue[0] if self.ready_queue else None

    def _finalize_ready_task(self, action):
        """Handle the current task using the chosen action."""
        task = self._current_task()
        if task is None:
            return 0.0, "idle", None, None

        if action is None or int(action) == self.wait_action:
            # Waiting scenario
            reward = self._calculate_reward(task, False, reason="wait")
            return reward, "wait", task, None

        decoded_action = self._decode_action(action)
        if decoded_action is None:
            self.ready_queue.popleft()
            self._reject_task(task)
            self.rejected_tasks += 1
            return self._calculate_reward(task, False), "rejected", task, "invalid_action"

        server_index, vm_index = decoded_action
        # Reject scenario
        if not self._can_place_task(task, server_index, vm_index):
            self.ready_queue.popleft()
            self._reject_task(task)
            self.rejected_tasks += 1
            return self._calculate_reward(task, False), "rejected", task, "resource_or_deadline_violation"

        # Schedule scenario
        self.ready_queue.popleft()
        self._allocate_task(task, server_index, vm_index)
        reward = self._calculate_reward(task, True)
        return reward, "scheduled", task, None

    def _reject_task(self, task):
        """Reject a task and mark it as not schedulable anymore."""
        task.status = "rejected"
        task.finish_time = self.current_time
        task.assigned_location = None

    def _advance_time(self):
        """Move the simulation forward by one time unit."""
        self.current_time += 1.0
        self._release_completed_tasks()

    def _count_running_tasks(self):
        """Count the number of currently running tasks."""
        count = 0
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                count += len(self.running_tasks[server_index][vm_index])
        return count

    def _is_done(self):
        """Check whether the episode is finished."""
        # The episode is done if there are no ready tasks, no running tasks, and all tasks have been handled (either completed or rejected).
        no_ready_tasks = len(self.ready_queue) == 0
        no_running_tasks = self._count_running_tasks() == 0
        all_tasks_handled = (self.completed_tasks + self.rejected_tasks) >= self.total_tasks
        return no_ready_tasks and no_running_tasks and all_tasks_handled

    def _get_resource_vector(self):
        """Flatten the resource matrix into a feature vector."""
        resource_features = []
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                resource_features.extend(self.resources[server_index][vm_index])
        return resource_features

    def _build_info(self, status, task=None, reward=0.0, action=None, reason=None):
        """Build the info dictionary returned by step()."""
        return {
            "status": status,
            "reason": reason,
            "action": action,
            "task": None if task is None else {
                "job_id": task.job_id,
                "task_index": task.task_index,
                "cpu_demand": task.cpu_demand,
                "ram_demand": task.ram_demand,
                "disk_demand": task.disk_demand,
                "runtime": task.runtime,
                "deadline": task.deadline,
            },
            "reward": reward,
            "completed_tasks": self.completed_tasks,
            "rejected_tasks": self.rejected_tasks,
            "running_tasks": self._count_running_tasks(),
            "ready_tasks": len(self.ready_queue),
            "current_time": self.current_time,
        }

    def get_state(self):
        """Return the current observation vector."""
        task = self._current_task()
        if task is None:
            task_features = [0.0] * 6
        else:
            wait_time = self._task_wait_time(task)
            task_features = [
                task.cpu_demand,
                task.ram_demand,
                task.disk_demand,
                task.runtime / self.max_runtime if self.max_runtime > 0 else 0.0,
                task.deadline / self.max_deadline if self.max_deadline > 0 else 0.0,
                wait_time / float(self.max_steps),
            ]

        summary_features = [
            self.current_time / float(self.max_steps),
            len(self.ready_queue) / float(max(1, self.total_tasks)),
            self._count_running_tasks() / float(max(1, self.total_tasks)),
        ]
        return np.array(summary_features + task_features + self._get_resource_vector(), dtype=np.float32)

    def get_valid_actions(self):
        """Return the actions that can currently schedule the ready task."""
        task = self._current_task()
        valid_actions = [self.wait_action]
        if task is None:
            return valid_actions

        for action_index, (server_index, vm_index) in enumerate(self.action_map):
            if self._can_place_task(task, server_index, vm_index):
                valid_actions.append(action_index)
        return valid_actions

    def reset(self, seed=None, options=None):
        """Reset the environment and return the initial observation."""
        super().reset(seed=seed)
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        self.tasks = copy.deepcopy(self.job_templates)
        self.resources = self._build_server_resources()
        self.running_tasks = self._build_running_task_slots()
        self.ready_queue = deque()
        self.current_time = 0.0
        self.completed_tasks = 0
        self.rejected_tasks = 0
        self.steps_taken = 0
        self.terminated = False
        self.truncated = False

        for task in self.tasks:
            task.status = "pending"
            task.ready_time = None
            task.start_time = None
            task.finish_time = None
            task.assigned_location = None
            task.ready_time = self.current_time
            task.status = "ready"
            self.ready_queue.append(task)

        self._release_completed_tasks()
        return self.get_state(), self._build_info("reset")

    def step(self, action):
        """Advance the environment by one decision step."""
        if self.terminated:
            return self.get_state(), 0.0, True, self.truncated, self._build_info("terminated", reward=0.0, action=action)
        if self.truncated:
            return self.get_state(), 0.0, False, True, self._build_info("truncated", reward=0.0, action=action)

        # First, release any tasks that have completed by now to free up resources.
        self._release_completed_tasks()
        # Then, handle the current ready task using the chosen action and compute the reward.
        reward, status, task, reason = self._finalize_ready_task(action)
        self._advance_time()
        self.steps_taken += 1

        if self.steps_taken >= self.max_steps and not self._is_done():
            self.truncated = True

        self.terminated = self._is_done()
        info = self._build_info(status, task=task, reward=reward, action=action, reason=reason)
        return self.get_state(), reward, self.terminated, self.truncated, info

    def render(self):
        """Print a short text summary of the current state."""
        print(
            "time={0:.1f}, ready={1}, running={2}, completed={3}, rejected={4}".format(
                self.current_time,
                len(self.ready_queue),
                self._count_running_tasks(),
                self.completed_tasks,
                self.rejected_tasks,
            )
        )

    def close(self):
        """Release environment resources."""
        return None

In [ ]:
env = CloudSchedulingEnv(scale="small", workload_path="dataset.txt", num_tasks=None, num_servers=20, num_vms_per_server=2)
state, info = env.reset(seed=42)
print("state_size={0}, action_space_size={1}".format(env.state_size, env.action_space_size))
print("initial_ready_tasks={0}".format(info["ready_tasks"]))
for _ in range(50):
    action = random.choice(env.get_valid_actions())
    state, reward, done, truncated, info = env.step(action)
    print("reward={0:.4f}, status={1}, done={2}, truncated={3}".format(reward, info["status"], done, truncated))

## Part 2: Student Implementation Section

### 2.1. TODO: Complete the Scheduler

In this section, students will add their own reinforcement learning or heuristic scheduling solution.
TODO:
- Define a scheduler or agent for `CloudSchedulingEnv`.
- Train the agent or policy on the CloudSchedulingEnv workload.
- Record reward, completion rate, rejection rate, and runtime.
- Compare the learned policy against a simple baseline.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Part 2 — Student Implementation
# Two RL algorithms: DQN (Algorithm 1) and PPO (Algorithm 2)
# System: 200 servers, 5 VMs/server  →  action_space = 1001
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ── Shared utility ────────────────────────────────────────────────────────────
def collect_metrics(env):
    """Compute average waiting time and completion time from finished tasks."""
    wait_times, comp_times = [], []
    for task in env.tasks:
        if task.status == "finished" and task.start_time is not None:
            wait_times.append(task.start_time - task.ready_time)
        if task.finish_time is not None and task.ready_time is not None:
            comp_times.append(task.finish_time - task.ready_time)
    avg_wait = float(np.mean(wait_times)) if wait_times else 0.0
    avg_comp = float(np.mean(comp_times)) if comp_times else 0.0
    return avg_wait, avg_comp


# ══════════════════════════════════════════════════════════════════════════════
# Algorithm 1 — DQN  (Deep Q-Network)
# ══════════════════════════════════════════════════════════════════════════════

class DQNNetwork(nn.Module):
    """Two-hidden-layer Q-network."""
    def __init__(self, state_size, action_size, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),     nn.ReLU(),
            nn.Linear(hidden, action_size),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    """Fixed-size experience replay buffer."""
    def __init__(self, capacity=50_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (np.array(s,  dtype=np.float32),
                np.array(a),
                np.array(r,  dtype=np.float32),
                np.array(ns, dtype=np.float32),
                np.array(d,  dtype=np.float32))

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    """DQN agent with valid-action masking and target network."""

    def __init__(self, state_size, action_size, lr=1e-3, gamma=0.99,
                 epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995,
                 batch_size=64, target_update=10):
        self.action_size   = action_size
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size    = batch_size
        self.target_update = target_update
        self._steps        = 0

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.policy_net = DQNNetwork(state_size, action_size).to(self.device)
        self.target_net = DQNNetwork(state_size, action_size).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer()
        self.loss_fn   = nn.SmoothL1Loss()

    def select_action(self, state, valid_actions):
        """Epsilon-greedy with valid-action masking."""
        if random.random() < self.epsilon:
            return random.choice(valid_actions)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q = self.policy_net(state_t).squeeze(0).cpu().numpy()
        masked = np.full(self.action_size, -np.inf)
        for a in valid_actions:
            masked[a] = q[a]
        return int(np.argmax(masked))

    def update(self):
        """One gradient-descent step on a sampled mini-batch."""
        if len(self.buffer) < self.batch_size:
            return None
        s, a, r, ns, d = self.buffer.sample(self.batch_size)
        s  = torch.FloatTensor(s).to(self.device)
        a  = torch.LongTensor(a).unsqueeze(1).to(self.device)
        r  = torch.FloatTensor(r).to(self.device)
        ns = torch.FloatTensor(ns).to(self.device)
        d  = torch.FloatTensor(d).to(self.device)

        q_cur = self.policy_net(s).gather(1, a).squeeze(1)
        with torch.no_grad():
            q_next = self.target_net(ns).max(1)[0]
        q_tgt = r + self.gamma * q_next * (1 - d)

        loss = self.loss_fn(q_cur, q_tgt)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        self._steps += 1
        if self._steps % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return loss.item()


# ══════════════════════════════════════════════════════════════════════════════
# Algorithm 2 — PPO  (Proximal Policy Optimisation)
# ══════════════════════════════════════════════════════════════════════════════

class PPONetwork(nn.Module):
    """Shared-backbone Actor-Critic network."""
    def __init__(self, state_size, action_size, hidden=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_size, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),     nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, action_size)
        self.critic = nn.Linear(hidden, 1)

    def forward(self, x):
        feat = self.shared(x)
        return self.actor(feat), self.critic(feat)


class PPOAgent:
    """PPO agent with clipped surrogate objective and entropy bonus."""

    def __init__(self, state_size, action_size, lr=3e-4, gamma=0.99,
                 clip_eps=0.2, epochs=4, batch_size=64):
        self.gamma      = gamma
        self.clip_eps   = clip_eps
        self.epochs     = epochs
        self.batch_size = batch_size
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net        = PPONetwork(state_size, action_size).to(self.device)
        self.optimizer  = optim.Adam(self.net.parameters(), lr=lr)
        # per-episode rollout buffers
        self._states, self._actions, self._rewards = [], [], []
        self._log_probs, self._values, self._dones  = [], [], []

    def select_action(self, state, valid_actions):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits, value = self.net(state_t)
        mask = torch.full((logits.shape[-1],), -1e9, device=self.device)
        for a in valid_actions:
            mask[a] = 0.0
        dist   = torch.distributions.Categorical(logits=logits.squeeze(0) + mask)
        action = dist.sample()
        self._states.append(state)
        self._actions.append(action.item())
        self._log_probs.append(dist.log_prob(action).item())
        self._values.append(value.item())
        return action.item()

    def store_reward(self, reward, done):
        self._rewards.append(reward)
        self._dones.append(done)

    def update(self):
        if len(self._rewards) < self.batch_size:
            return None
        # Monte-Carlo returns
        returns, G = [], 0.0
        for r, d in zip(reversed(self._rewards), reversed(self._dones)):
            G = r + self.gamma * G * (1 - d)
            returns.insert(0, G)

        returns = torch.FloatTensor(returns).to(self.device)
        advs    = returns - torch.FloatTensor(self._values).to(self.device)
        advs    = (advs - advs.mean()) / (advs.std() + 1e-8)
        states  = torch.FloatTensor(np.array(self._states)).to(self.device)
        actions = torch.LongTensor(self._actions).to(self.device)
        old_lp  = torch.FloatTensor(self._log_probs).to(self.device)

        losses = []
        for _ in range(self.epochs):
            logits, values = self.net(states)
            dist   = torch.distributions.Categorical(logits=logits)
            new_lp = dist.log_prob(actions)
            ratio  = (new_lp - old_lp).exp()
            l_clip = -torch.min(ratio * advs,
                                torch.clamp(ratio, 1-self.clip_eps, 1+self.clip_eps) * advs).mean()
            l_vf   = nn.MSELoss()(values.squeeze(-1), returns)
            l_ent  = -dist.entropy().mean()
            loss   = l_clip + 0.5 * l_vf + 0.01 * l_ent
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
            self.optimizer.step()
            losses.append(loss.item())

        # clear rollout buffers
        self._states, self._actions, self._rewards = [], [], []
        self._log_probs, self._values, self._dones  = [], [], []
        return float(np.mean(losses))


# ══════════════════════════════════════════════════════════════════════════════
# Training loop
# ══════════════════════════════════════════════════════════════════════════════

def train(agent, env, num_episodes=200, is_ppo=False, label="Agent"):
    reward_hist, loss_hist = [], []
    rej_hist, wait_hist, comp_hist = [], [], []

    for ep in range(num_episodes):
        state, _ = env.reset(seed=ep)
        total_reward, losses = 0.0, []
        done = truncated = False

        while not (done or truncated):
            valid = env.get_valid_actions()
            action = agent.select_action(state, valid)
            next_state, reward, done, truncated, info = env.step(action)

            if is_ppo:
                agent.store_reward(reward, float(done or truncated))
            else:
                agent.buffer.push(state, action, reward, next_state,
                                  float(done or truncated))
                loss = agent.update()
                if loss is not None:
                    losses.append(loss)

            state = next_state
            total_reward += reward

        if is_ppo:
            loss = agent.update()
            if loss is not None:
                losses.append(loss)

        total       = env.total_tasks
        rejected    = info["rejected_tasks"]
        rej_rate    = rejected / total if total > 0 else 0.0
        avg_wait, avg_comp = collect_metrics(env)

        reward_hist.append(total_reward)
        loss_hist.append(float(np.mean(losses)) if losses else 0.0)
        rej_hist.append(rej_rate)
        wait_hist.append(avg_wait)
        comp_hist.append(avg_comp)

        if (ep + 1) % 10 == 0:
            print(f"[{label}] ep {ep+1:4d} | reward {total_reward:8.2f} | "
                  f"completed {info['completed_tasks']}/{total} | "
                  f"reject {rej_rate*100:.1f}% | "
                  f"wait {avg_wait:.2f} | comp {avg_comp:.2f}")

    return reward_hist, loss_hist, rej_hist, wait_hist, comp_hist


# ── Instantiate environments (200 servers, 5 VMs — as per lab spec) ──────────
NUM_TASKS_TRAIN = 500   # use a subset for faster iteration; set None for full dataset

env_dqn = CloudSchedulingEnv(
    workload_path="dataset.txt",
    num_tasks=NUM_TASKS_TRAIN,
    num_servers=200,
    num_vms_per_server=5,
)
env_ppo = CloudSchedulingEnv(
    workload_path="dataset.txt",
    num_tasks=NUM_TASKS_TRAIN,
    num_servers=200,
    num_vms_per_server=5,
)

print(f"state_size={env_dqn.state_size}, action_size={env_dqn.action_space_size}, "
      f"total_tasks={env_dqn.total_tasks}")

# ── Train DQN ─────────────────────────────────────────────────────────────────
dqn_agent = DQNAgent(
    state_size=env_dqn.state_size,
    action_size=env_dqn.action_space_size,
    lr=1e-3, gamma=0.99,
    epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.99,
    batch_size=64, target_update=10,
)

print("\n=== Training DQN ===")
dqn_rewards, dqn_losses, dqn_rej, dqn_wait, dqn_comp = train(
    dqn_agent, env_dqn, num_episodes=200, is_ppo=False, label="DQN")

# ── Train PPO ─────────────────────────────────────────────────────────────────
ppo_agent = PPOAgent(
    state_size=env_ppo.state_size,
    action_size=env_ppo.action_space_size,
    lr=3e-4, gamma=0.99,
    clip_eps=0.2, epochs=4, batch_size=64,
)

print("\n=== Training PPO ===")
ppo_rewards, ppo_losses, ppo_rej, ppo_wait, ppo_comp = train(
    ppo_agent, env_ppo, num_episodes=200, is_ppo=True, label="PPO")


# ══════════════════════════════════════════════════════════════════════════════
# Plot training curves
# ══════════════════════════════════════════════════════════════════════════════

def smooth(arr, w=10):
    return np.convolve(arr, np.ones(w) / w, mode="valid")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("DQN vs PPO — Training Metrics (200 servers, 5 VMs)", fontsize=14)

axes[0, 0].plot(smooth(dqn_rewards), label="DQN", color="steelblue")
axes[0, 0].plot(smooth(ppo_rewards), label="PPO", color="darkorange")
axes[0, 0].set_title("Reward Curve"); axes[0, 0].set_xlabel("Episode")
axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(smooth(dqn_losses), label="DQN", color="steelblue")
axes[0, 1].plot(smooth(ppo_losses), label="PPO", color="darkorange")
axes[0, 1].set_title("Loss Curve"); axes[0, 1].set_xlabel("Episode")
axes[0, 1].legend(); axes[0, 1].grid(True)

axes[1, 0].plot(smooth(dqn_rej), label="DQN", color="steelblue")
axes[1, 0].plot(smooth(ppo_rej), label="PPO", color="darkorange")
axes[1, 0].set_title("Rejection Rate"); axes[1, 0].set_xlabel("Episode")
axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].plot(smooth(dqn_wait), label="DQN Wait",  color="steelblue",  linestyle="-")
axes[1, 1].plot(smooth(ppo_wait), label="PPO Wait",  color="darkorange", linestyle="-")
axes[1, 1].plot(smooth(dqn_comp), label="DQN Comp",  color="steelblue",  linestyle="--")
axes[1, 1].plot(smooth(ppo_comp), label="PPO Comp",  color="darkorange", linestyle="--")
axes[1, 1].set_title("Avg Waiting & Completion Time"); axes[1, 1].set_xlabel("Episode")
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation on testset.txt
# ══════════════════════════════════════════════════════════════════════════════

def evaluate(agent, workload_path, num_servers=200, num_vms=5,
             is_ppo=False, label="Agent", num_tasks=None):
    env_test = CloudSchedulingEnv(
        workload_path=workload_path,
        num_tasks=num_tasks,
        num_servers=num_servers,
        num_vms_per_server=num_vms,
    )
    state, _ = env_test.reset(seed=0)

    if not is_ppo:
        agent.epsilon = 0.0   # greedy policy

    done = truncated = False
    total_reward = 0.0
    while not (done or truncated):
        valid = env_test.get_valid_actions()
        with torch.no_grad():
            action = agent.select_action(state, valid)
        state, reward, done, truncated, info = env_test.step(action)
        total_reward += reward

    total      = env_test.total_tasks
    rejected   = info["rejected_tasks"]
    completed  = info["completed_tasks"]
    rej_rate   = rejected / total if total > 0 else 0.0
    avg_wait, avg_comp = collect_metrics(env_test)

    print(f"\n=== {label} — Testset Evaluation ===")
    print(f"  Total tasks     : {total}")
    print(f"  Completed       : {completed}  ({100*(1-rej_rate):.1f}%)")
    print(f"  Rejected        : {rejected}  ({rej_rate*100:.1f}%)")
    print(f"  Avg Wait Time   : {avg_wait:.4f}")
    print(f"  Avg Comp Time   : {avg_comp:.4f}")
    print(f"  Total Reward    : {total_reward:.2f}")

    return dict(completed=completed, rejected=rejected, reject_rate=rej_rate,
                avg_wait=avg_wait, avg_comp=avg_comp, total_reward=total_reward)

NUM_TASKS_TEST = 500   # subset of testset for evaluation; set None for full

dqn_metrics = evaluate(dqn_agent, "testset.txt",
                        is_ppo=False, label="DQN", num_tasks=NUM_TASKS_TEST)
ppo_metrics = evaluate(ppo_agent, "testset.txt",
                        is_ppo=True,  label="PPO", num_tasks=NUM_TASKS_TEST)

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"{'Metric':<25} {'DQN':>12} {'PPO':>12}")
print("="*55)
for key in ["reject_rate", "avg_wait", "avg_comp", "total_reward"]:
    print(f"{key:<25} {dqn_metrics[key]:>12.4f} {ppo_metrics[key]:>12.4f}")
print("="*55)


---
# CONGRATULATIONS!

This Lab 06 notebook now provides a complete cloud scheduling setup for reinforcement learning studies:
- a Gymnasium-style `CloudSchedulingEnv` for resource provisioning and task scheduling
- a clear workload model, state design, action space, and reward structure
- dedicated student implementation placeholders for scheduling algorithms and evaluation
- a shared environment for comparing rule-based and learned scheduling policies

Technical takeaways:
- resource-aware scheduling depends on state quality, action constraints, and reward shaping
- a valid Gymnasium API makes the environment easier to test, train, and compare
- evaluation should consider reward, completion rate, rejection rate, and runtime behavior

## References

- Inspired by Shahid Mohammed Shaikbepari in Deep-Reinforcement-Learning-for-cloud repository
- Gymnasium documentation: https://gymnasium.farama.org/
- Stable-Baselines3 documentation: https://stable-baselines3.readthedocs.io/

---

## ADDITIONAL INFORMATION

**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn

**Last Updated**: May, 2026